# Exercise 2 — Inspect InstaNovo predictions & assemble nanobody nb6 with InstaNexus

**27666 · June 2026 · Module 2**

**Goal**: take real mass-spectrometry data from a nanobody experiment, look at what InstaNovo predicted at the peptide level, then reconstruct the full nanobody sequence using the **InstaNexus** assembly pipeline.

**Data**: `NB6.csv` — InstaNovo predictions on a nanobody (**nb6**) digested by 9 different proteases (Trypsin, Chymotrypsin, GluC, Elastase, ProteinaseK, Thermolysin, Papain, Krakatoa, Vesuvius) and measured by LC-MS/MS.

**Pipeline you'll run**: confidence filtering → multi-protease assembly (greedy + De Bruijn graph) → MMseqs2 clustering → Clustal Omega alignment → consensus scaffold → comparison vs. the **known reference sequence** of nb6.

**Runtime**: ~15 min on free Colab. Some of that is the binary install.

**Inputs**: `NB6.csv` is fetched from the course repo (`Day_5/morning/data/`) by `wget` — no manual upload required. The metadata JSON and contaminants FASTA are generated inline by the notebook.

> **Hyperparameter note**: the InstaNexus README example uses `--conf 0.9 --size-threshold 12`, which is tuned for BSA (~600 aa, deep coverage). For a ~150-aa nanobody with limited peptide redundancy we relax to `--conf 0.8 --size-threshold 8`. Sweeping these is one of the discussion prompts at the end.

## 🧑‍🎓 Student notebook — fill in the blanks!

This notebook contains **8 `TODO_N` markers** where you need to write a small piece of code before the cell will run. Each `TODO` is a real design choice a researcher makes — not a trick. If you get stuck, the markdown above the cell and the hint in the comment usually contain the answer.

**The blanks**:

- `TODO_1` (Sec. 5+6) — InstaNovo confidence cutoff `--conf` (applied to both greedy + DBG runs).
- `TODO_2` (Sec. 5+6) — minimum peptide overlap `--min-overlap`.
- `TODO_3` (Sec. 5+6) — minimum peptides supporting a contig `--size-threshold`.
- `TODO_4` (Sec. 6) — DBG k-mer length `--kmer-size`.
- `TODO_5` (Sec. 8) — CDR1 boundary positions in the nb6 reference.
- `TODO_6` (Sec. 8) — CDR3 start position and fallback end position.
- `TODO_7` (Sec. 8) — gap-open and gap-extend penalties for scaffold→reference alignment.
- `TODO_8` (Sec. 10) — L2 normalisation step that turns dot product into cosine similarity.

Cells run top-to-bottom. If a cell crashes with `SyntaxError` or `NameError`, check for an unfilled `____` or `___`. A `_solved` companion notebook exists — try to resist looking at it!


## Attribution

- **InstaNovo** (de novo peptide sequencing): Eloff K.*, Kalogeropoulos K.* et al. *Nat Mach Intell* 7, 565–579 (2025).
- **InstaNexus** (assembly pipeline): Reverenna M. et al. *Mol Cell Proteomics* 25:4 (2026). Repo: <https://github.com/Multiomics-Analytics-Group/InstaNexus>.
- **MMseqs2** (clustering): Steinegger & Söding *Nat Biotech* 35, 1026–1028 (2017).
- **Clustal Omega** (MSA): Sievers et al. *Mol Syst Biol* 7, 539 (2011).
- Nanobody nb6 reference sequence: from `InstaNexus-main/json/sample_metadata.json` (DTU Biosustain / Bioengineering).
- Raw MS data: DTU Bioengineering Proteomics Core Facility, 2025.

## 1. Setup

Install **InstaNexus** + binaries (MMseqs2, Clustal Omega). The first run takes ~2 min on Colab.

In [ ]:
# --- Python packages ---
# Pin pandas to exactly Colab's bundled version to avoid resolver complaints
# from google-colab / gradio / db-dtypes.
# We also install logomaker explicitly — the instanexus.consensus module imports it,
# but it isn't always pulled in as a transitive dep, leading to ModuleNotFoundError.
!pip install -q 'pandas==2.2.2' instanexus logomaker biopython matplotlib seaborn

# --- Clustal Omega via apt (Colab Ubuntu) ---
!apt-get install -y -qq clustalo > /dev/null

# --- MMseqs2 static binary ---
import os, subprocess, shutil
if shutil.which('mmseqs') is None:
    !wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /tmp/mmseqs.tar.gz
    !tar xzf /tmp/mmseqs.tar.gz -C /tmp/
    os.environ['PATH'] = '/tmp/mmseqs/bin:' + os.environ['PATH']

print('clustalo :', shutil.which('clustalo'))
print('mmseqs   :', shutil.which('mmseqs'))
!instanexus --help | head -5

# >>> NOTE: if pandas was already imported in this session, restart the runtime now
# (Runtime → Restart session) so the pinned version is picked up, then re-run from cell 1.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context('notebook')
WORK = Path('/content/27666_exercise2') if Path('/content').exists() else Path('exercise2_work')
WORK.mkdir(exist_ok=True)
print(f'working dir: {WORK}')

## 2. Load `NB6.csv` (InstaNovo predictions for nanobody nb6)

If you are on Colab, upload `NB6.csv` via the Files panel (or use the cell below). If you're running locally, edit the path.

In [ ]:
# Sample we're working on — single line to swap between nb1/nb6/etc.
SAMPLE_NAME = 'nb6'

# Pull NB6.csv from the course repo and save it lowercase so that
# Path(input_csv).stem == 'nb6' — that's how InstaNexus matches the
# sample_metadata.json key. Mismatched case → preprocessing fails silently.
NB_URL = 'https://raw.githubusercontent.com/DigBioLab/27666_Protein_Design/plm-and-foundation-models/Day_5/morning/data/NB6.csv'
NB_PATH = Path(f'{SAMPLE_NAME}.csv')   # lowercase on disk

if not NB_PATH.exists():
    !wget -q $NB_URL -O $NB_PATH

# Fallback: manual upload via Colab Files panel
# from google.colab import files
# uploaded = files.upload()
# NB_PATH = Path(next(iter(uploaded.keys())))

assert NB_PATH.exists() and NB_PATH.stat().st_size > 1_000_000, (
    f'{NB_PATH} missing or suspiciously small — check the wget URL or upload manually.'
)

df = pd.read_csv(NB_PATH)
print(f'sample: {SAMPLE_NAME}   file: {NB_PATH}   rows: {len(df):,}')
df.head()

## 3. Inspect the predictions

Columns:
- `experiment_name` — encodes the protease used in that run
- `scan_number` — index of the MS/MS scan
- `preds` — predicted peptide sequence (empty if InstaNovo had no confident prediction)
- `log_probs` — InstaNovo's log-probability of the prediction (sentinel `-1.0` means "no prediction")

In [ ]:
# Reference sequence of nb6 (from InstaNexus-main/json/sample_metadata.json).
# Defined here — before the pre-assembly checks — so the coverage cell
# can use it. To run a different sample, change SAMPLE_NAME above and
# replace this sequence.
NB_REF = ('QVQLQESGGGLVQPGGSLRLSCTASLNIFSINAMGWYRQAPGKQRELVAAITSGGSTNYADSVKGRFTISRDNAKSTVY'
          'LQMNSLKPEDTAVYYCHAEGPFNIATKEQYDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH')
print(f'reference {SAMPLE_NAME}: {len(NB_REF)} aa')

In [ ]:
# Parse the protease out of the experiment name
def protease_of(name: str) -> str:
    m = re.search(r'Nanobodies_([A-Za-z]+)_\d+ng', name)
    return m.group(1) if m else 'unknown'

df['protease'] = df['experiment_name'].apply(protease_of)
df['has_pred'] = df['preds'].notna() & (df['preds'] != '')

summary = (df.groupby('protease')
             .agg(scans=('scan_number', 'count'),
                  with_pred=('has_pred', 'sum'),
                  hit_rate=('has_pred', 'mean'))
             .sort_values('scans', ascending=False))
print(summary)

In [ ]:
# log_probs distribution for real predictions only
real = df[df['has_pred'] & (df['log_probs'] > -1)]   # -1 sentinel filtered out

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(real['log_probs'], bins=60, ax=axes[0], color='steelblue')
axes[0].set(title='InstaNovo log-prob distribution (predicted peptides)',
             xlabel='log P', ylabel='# peptides')

sns.boxplot(data=real, x='protease', y='log_probs', ax=axes[1],
             order=summary.index.tolist(), color='lightsteelblue')
axes[1].set(title='Confidence by protease', xlabel='', ylabel='log P')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

**Observations to make**:
- A large fraction of scans have **no prediction** — that's normal: not every MS/MS scan is fragmenting a peptide cleanly.
- Confidence varies between proteases. Why? (Hint: tryptic peptides are over-represented in InstaNovo's training data.)
- The `log_probs` are *raw model log-likelihoods*, not calibrated probabilities. This is the FDR problem we mentioned on Module 2 slide 15.

## 4. Pre-assembly checks

Before running the full pipeline, get a feel for the data. These checks tell us whether the assembly is likely to succeed, and which proteases are pulling their weight.

We will look at:
- **Per-residue confidence** on a 0–1 scale (instead of the raw `log_probs` sum).
- **Peptide length** by protease.
- **Peptide overlap** between proteases (Jaccard similarity).
- **Pre-assembly coverage** by exact substring match against the nb6 reference (this is the assembly-free baseline).


In [ ]:
# log_probs is the SUM of per-position log-probabilities over the peptide.
# For a comparable 0-1 scale, take the geometric mean per residue: exp(log_prob / L).
real = real.copy()  # avoid SettingWithCopyWarning
real['pep_len']     = real['preds'].str.len()
real['conf_per_aa'] = np.exp(real['log_probs'] / real['pep_len'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(real['conf_per_aa'], bins=60, ax=axes[0], color='steelblue')
axes[0].axvline(0.8, color='red', linestyle='--', label='`--conf 0.8` cutoff')
axes[0].set(title='Per-residue confidence (0-1 scale)',
            xlabel='exp(log_p / L)', ylabel='# peptides')
axes[0].legend()

sns.boxplot(data=real, x='protease', y='conf_per_aa', ax=axes[1],
            order=summary.index.tolist(), color='lightsteelblue')
axes[1].axhline(0.8, color='red', linestyle='--')
axes[1].set(title='Per-residue confidence by protease', xlabel='', ylabel='exp(log_p / L)')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

n_hi = (real['conf_per_aa'] > 0.8).sum()
print(f'Peptides with per-residue confidence > 0.8: {n_hi} ({n_hi/len(real):.1%} of confident predictions)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=real, x='protease', y='pep_len', ax=ax,
            order=summary.index.tolist(), color='peachpuff')
ax.set(title='Peptide length distribution by protease', xlabel='', ylabel='peptide length (aa)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

print('Median peptide length per protease:')
print(real.groupby('protease')['pep_len'].median().sort_values(ascending=False))

In [ ]:
# Jaccard similarity of peptide sets between protease pairs.
# We WANT low off-diagonal Jaccard: different proteases should give complementary peptides.
peps_by_protease = {p: set(real[real.protease == p].preds.unique())
                    for p in real.protease.unique()}
ps = sorted(peps_by_protease.keys())
n = len(ps)
J = np.zeros((n, n))
for i, p1 in enumerate(ps):
    for j, p2 in enumerate(ps):
        a, b = peps_by_protease[p1], peps_by_protease[p2]
        J[i, j] = len(a & b) / max(1, len(a | b))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(J, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=ps, yticklabels=ps, ax=ax,
            cbar_kws={'label': 'Jaccard similarity'})
ax.set_title('Peptide overlap between proteases')
plt.xticks(rotation=35); plt.tight_layout(); plt.show()

union = set().union(*peps_by_protease.values())
print(f'\nTotal unique peptides across all proteases: {len(union)}')
for p in ps:
    print(f'  {p:14s}: {len(peps_by_protease[p]):4d} unique peptides')

In [ ]:
# Quick coverage estimate: for each high-confidence peptide (>0.8 per-residue),
# count exact substring matches against the nb6 reference. This is the
# assembly-free LOWER BOUND for what InstaNexus will recover.
HIGH_CONF = real[real['conf_per_aa'] > 0.8]
covered_pre = np.zeros(len(NB_REF), dtype=int)
matched_peptides = 0
for pep in HIGH_CONF['preds'].unique():
    start, found = 0, False
    while True:
        idx = NB_REF.find(pep, start)
        if idx < 0: break
        covered_pre[idx:idx + len(pep)] += 1
        start = idx + 1
        found = True
    if found:
        matched_peptides += 1

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(NB_REF)), covered_pre, width=1.0, color='steelblue')
ax.set(xlabel=f'position in {SAMPLE_NAME} reference', ylabel='# peptide matches',
       title=f'Pre-assembly coverage by exact-match high-confidence peptides')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

frac = (covered_pre > 0).mean()
n_uniq_high = HIGH_CONF['preds'].nunique()
print(f'\nHigh-confidence unique peptides: {n_uniq_high}')
print(f'  ... of which match the {SAMPLE_NAME} reference exactly: {matched_peptides}')
print(f'Reference positions covered by ≥1 high-confidence peptide: '
      f'{(covered_pre > 0).sum()} / {len(NB_REF)} ({frac:.1%})')
print('\nThis is a LOWER BOUND: InstaNexus also handles inexact matches and chains '
      'peptides through overlapping ends.')

**What to look at**:

- **Per-residue confidence**: how many peptides survive `--conf 0.8`? If very few, we should relax the cutoff.
- **Peptide length**: Trypsin / LysC give short, regular peptides; ProteinaseK / Elastase give a wider distribution. Mixing proteases is how we cover regions that one protease misses.
- **Overlap heatmap**: high off-diagonal Jaccard means two proteases are redundant. We *want* low off-diagonal values so the panel covers more of the protein.
- **Pre-assembly coverage**: rough preview of where assembly will succeed. If a region is dark here, InstaNexus is unlikely to recover it — and CDR3 is almost always the dark spot.


## 5. Set up files for InstaNexus

We need three input files in the working directory:
1. The InstaNovo CSV (`NB6.csv`)
2. A **metadata JSON** that tells InstaNexus the reference protein + which proteases were used
3. A **contaminants FASTA** (cell-line / column / keratin contaminants to filter out)

We construct minimal versions of (2) and (3) inline so the notebook is self-contained.

In [ ]:
# NB_REF was defined earlier (just after the 'Inspect predictions' header) so the
# pre-assembly checks could use it. Re-using the same constant here for the JSON.

METADATA = {
    SAMPLE_NAME: [{
        'protein': NB_REF,
        'chain': '',
        'proteases': ['Vesuvius', 'Krakatoa', 'Elastase', 'Trypsin', 'GluC',
                      'Chymotrypsin', 'Papain', 'ProteinaseK', 'Thermolysin'],
    }]
}
META_PATH = WORK / 'metadata.json'
META_PATH.write_text(json.dumps(METADATA, indent=2))

# Minimal contaminants FASTA — common keratins, trypsin self-digest, BSA
CONTAM_FASTA = '''>Trypsin_BOVIN_self
VVGGYTCAANSIPYQVSLNSGSHFCGGSLINSQWVVSAAHCYKSRIQVRLGEHNIDVLEGNEQFINAAKIIKHPNFDR
>Keratin_K1_HUMAN_frag
MSRQFSSRSGYRSGGGFSSGSAGIINYQRRTTSSSTRRSGGGGGRFSSCGGGGGSFGAGGGFGSRSLVNLGGSKSISIS
>BSA_BOVIN_frag
MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEEHFKGLVLIAFSQYLQQCPFDEHVKLVNELTEFAKTCVADESHAGCEK
'''
CONTAM_PATH = WORK / 'contaminants.fasta'
CONTAM_PATH.write_text(CONTAM_FASTA)

# Copy the input CSV into the working dir
import shutil
INPUT_CSV = WORK / f'{SAMPLE_NAME.upper()}.csv'
if INPUT_CSV.resolve() != NB_PATH.resolve():
    shutil.copy(NB_PATH, INPUT_CSV)

print('=== Files prepared for the InstaNexus pipeline ===')
print(f'sample        : {SAMPLE_NAME}')
print(f'metadata.json : {META_PATH}')
print(f'contaminants  : {CONTAM_PATH}')
print(f'input CSV     : {INPUT_CSV}')
print(f'reference     : {len(NB_REF)} aa  ({NB_REF[:40]}…)')
print()
print('All three files were created inside the notebook just now — no manual setup needed.')
print('The instanexus CLI will pick them up from the paths above.')

## 6. Run InstaNexus — greedy assembly

Greedy mode chains peptides by exact suffix-prefix overlap. Fast, easy to interpret.

Parameters (relaxed for a small ~150-aa nanobody, see the note in the cover cell):
- `--conf 0.8` — keep peptides scoring above this fraction of InstaNovo's confidence
- `--min-overlap 3` — peptides must share at least 3 residues to chain
- `--size-threshold 8` — a contig is kept if it's backed by at least 8 peptides

Outputs land under `outputs/<sample>/greedy_…/`.

In [ ]:
import subprocess
OUT_DIR = WORK / 'outputs'

greedy_cmd = [
    'instanexus',
    '--input-csv', str(INPUT_CSV),
    '--folder-outputs', str(OUT_DIR),
    '--metadata-json-path', str(META_PATH),
    '--contaminants-fasta-path', str(CONTAM_PATH),
    '--assembly-mode', 'greedy',
    '--conf', '___',    # TODO_1: InstaNovo confidence cutoff. The InstaNexus README uses 0.9 for BSA (~600 aa, high redundancy). nb6 is ~150 aa — should you be stricter or more permissive? (try 0.8)
    '--min-overlap', '_',    # TODO_2: minimum amino-acid overlap to chain two peptides in greedy mode. Trade-off: 2 → many false joins; 5 → very few chains. Try 3.
    '--size-threshold', '_',    # TODO_3: minimum number of peptides supporting a contig. BSA default is 12; for a 150-aa nanobody with limited redundancy, try 8.
    '--min-seq-id', '0.85',
    '--coverage', '0.8',
]
print('$', ' '.join(greedy_cmd))
import time
t0 = time.time()
result = subprocess.run(greedy_cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1500:])
    print(f'\n✗ Greedy assembly FAILED after {elapsed:.1f}s — see STDERR above.')
else:
    print(f'\n✓ Greedy assembly complete in {elapsed:.1f}s.')

## 7. Run InstaNexus — De Bruijn graph assembly

DBG mode breaks peptides into **k-mers**, builds a graph of overlaps at fixed k, and walks paths through it. More robust to small errors than greedy. We use k = 7.

In [ ]:
dbg_cmd = [
    'instanexus',
    '--input-csv', str(INPUT_CSV),
    '--folder-outputs', str(OUT_DIR),
    '--metadata-json-path', str(META_PATH),
    '--contaminants-fasta-path', str(CONTAM_PATH),
    '--assembly-mode', 'dbg',
    '--conf', '___',    # TODO_1: InstaNovo confidence cutoff. The InstaNexus README uses 0.9 for BSA (~600 aa, high redundancy). nb6 is ~150 aa — should you be stricter or more permissive? (try 0.8)
    '--kmer-size', '_',    # TODO_4: De Bruijn k-mer length. Short k (3-5) = sensitive but more ambiguous walks; long k (9-11) = stricter but sparser graph. Try 7.
    '--size-threshold', '_',    # TODO_3: minimum number of peptides supporting a contig. BSA default is 12; for a 150-aa nanobody with limited redundancy, try 8.
    '--min-overlap', '_',    # TODO_2: minimum amino-acid overlap to chain two peptides in greedy mode. Trade-off: 2 → many false joins; 5 → very few chains. Try 3.
    '--min-seq-id', '0.85',
    '--coverage', '0.8',
]
print('$', ' '.join(dbg_cmd))
import time
t0 = time.time()
result = subprocess.run(dbg_cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1500:])
    print(f'\n✗ DBG assembly FAILED after {elapsed:.1f}s — see STDERR above.')
else:
    print(f'\n✓ DBG assembly complete in {elapsed:.1f}s.')

## 8. Find the consensus scaffolds

Each run wrote a timestamped output folder. Pick up the FASTA of consensus sequences.

In [ ]:
from Bio import SeqIO

# InstaNexus writes outputs at:
#   <folder_outputs>/<csv_stem>/<mode>_c<conf>[_ks<k>]/
#       ├── cleaned.csv
#       └── scaffolds/
#           ├── scaffolds.fasta                        ← initial contigs
#           └── consensus/
#               └── consensus_fasta/
#                   └── *_consensus.fasta              ← FINAL consensus scaffolds
# We use Path(INPUT_CSV).stem as the run-level folder name so this matches whatever
# we passed as --input-csv.

RUN_NAME = Path(INPUT_CSV).stem   # 'nb6'

def list_runs(mode_prefix: str):
    '''Timestamped param-run subfolders for a given assembly mode, newest first.'''
    base = OUT_DIR / RUN_NAME
    if not base.exists():
        return []
    return sorted(base.glob(f'{mode_prefix}*'), key=lambda p: p.stat().st_mtime, reverse=True)

def load_consensus(run_dir: Path) -> dict:
    '''Try the final consensus FASTAs first; fall back to scaffolds.fasta.'''
    consensus_dir = run_dir / 'scaffolds' / 'consensus' / 'consensus_fasta'
    fastas = sorted(consensus_dir.glob('*_consensus.fasta')) if consensus_dir.exists() else []
    if not fastas:
        fallback = run_dir / 'scaffolds' / 'scaffolds.fasta'
        if fallback.exists():
            fastas = [fallback]
    seqs = {}
    for fa in fastas:
        for rec in SeqIO.parse(fa, 'fasta'):
            seqs[f'{fa.stem}::{rec.id}'] = str(rec.seq)
    return seqs

greedy_runs = list_runs('greedy')
dbg_runs    = list_runs('dbg')
print(f'Looking under: {OUT_DIR / RUN_NAME}')
print(f'greedy runs found: {len(greedy_runs)}  →  {[r.name for r in greedy_runs]}')
print(f'dbg    runs found: {len(dbg_runs)}  →  {[r.name for r in dbg_runs]}')

greedy_seqs = load_consensus(greedy_runs[0]) if greedy_runs else {}
dbg_seqs    = load_consensus(dbg_runs[0])    if dbg_runs    else {}
print(f'\ngreedy scaffolds: {len(greedy_seqs)}')
for k, v in list(greedy_seqs.items())[:5]:
    print(f'  {k}  len={len(v)}  {v[:50]}…')
print(f'\ndbg scaffolds: {len(dbg_seqs)}')
for k, v in list(dbg_seqs.items())[:5]:
    print(f'  {k}  len={len(v)}  {v[:50]}…')



## 9. Map each scaffold onto the nb6 reference

Global pairwise alignment of each scaffold against the known nb6 sequence, then we plot a **coverage track** and call out the CDR regions.

In [ ]:
from Bio import pairwise2

# Approximate CDR boundaries for VHH (Kabat-ish numbering applied to the framework). These are *approximate*
# and meant for visual interpretation — for production work use ANARCI or IMGT.
CDR1 = (__, __)    # TODO_5: CDR1 of a VHH framework starts around position 25 and ends around position 35 (Kabat-ish numbering). Fill the boundaries.
CDR2 = (49, 58)
WGQ  = NB_REF.find('WGQGTQ')
CDR3 = (__, WGQ if WGQ > 0 else ___)    # TODO_6: CDR3 begins right after the canonical C-anchor (second Cys ~ position 95) and ends just before the WGQGTQ motif. Fill in both numbers — the second one is a fallback length used if WGQGTQ is not found.

def best_alignment(query: str, target: str = NB_REF):
    aln = pairwise2.align.localxs(target, query, ___, ___, one_alignment_only=True)    # TODO_7: gap-open and gap-extend penalties (both negative — they penalise gaps). -2 and -1 are reasonable defaults; make them more negative to discourage gaps further.
    return aln[0] if aln else None

def coverage_mask(seqs: dict) -> np.ndarray:
    covered = np.zeros(len(NB_REF), dtype=int)
    for q in seqs.values():
        a = best_alignment(q)
        if a is None:
            continue
        target_aln, query_aln, score, start, end = a
        # Walk through the aligned region and increment coverage where query has a non-gap and matches
        ti = 0
        for tc, qc in zip(target_aln, query_aln):
            if tc != '-':
                if qc != '-' and ti >= start and ti < end:
                    covered[ti] += 1
                ti += 1
    return covered

greedy_cov = coverage_mask(greedy_seqs)
dbg_cov    = coverage_mask(dbg_seqs)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5.5), sharex=True)
for ax, cov, label, color in zip(axes,
                                  [greedy_cov, dbg_cov],
                                  ['Greedy', 'DBG (k=7)'],
                                  ['#2ca02c', '#1f77b4']):
    ax.bar(range(len(NB_REF)), cov, color=color, width=1.0, alpha=0.9)
    ax.set_ylabel(f'{label}\n# scaffolds')
    ymax = max(1, ax.get_ylim()[1])
    for cdr, name in zip([CDR1, CDR2, CDR3], ['CDR1', 'CDR2', 'CDR3']):
        ax.axvspan(cdr[0], cdr[1], color='red', alpha=0.12)
        ax.text((cdr[0]+cdr[1])/2, ymax*0.9, name, ha='center', fontsize=9, color='darkred')
    ax.spines[['top', 'right']].set_visible(False)
axes[1].set_xlabel(f'position in {SAMPLE_NAME} reference')
plt.suptitle(f'InstaNexus coverage of {SAMPLE_NAME} vs. reference', y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig(WORK/f'{SAMPLE_NAME}_coverage.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'\nReference length             : {len(NB_REF)} aa')
print(f'Greedy positions covered ≥1× : {int((greedy_cov > 0).sum())}  ({(greedy_cov > 0).mean():.1%})')
print(f'DBG    positions covered ≥1× : {int((dbg_cov > 0).sum())}  ({(dbg_cov > 0).mean():.1%})')

**Reading the coverage track**:
- Tall bars = many scaffolds cover this position → high redundancy → high confidence in the residue identity
- Drops to zero in the CDR regions (red bands) = the hardest part of the problem. CDR3 in particular is the canonical InstaNexus failure mode.
- Drops to zero outside CDRs = something is wrong (low signal protease, missed cleavage, contaminant, etc.)

**Real takeaway**: this same workflow has been used to reconstruct therapeutic antibodies and nanobodies *with no reference at all* in the Reverenna et al. (2026) paper.

## 10. Best scaffold vs. reference — identity over the variable domain

In [ ]:
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

def best_pick(seqs: dict):
    """Return (name, seq, identity_over_variable_domain, alignment) of the scaffold with
    highest identity to the nanobody variable domain (residues 0..WGQ)."""
    variable = NB_REF[:WGQ if WGQ > 0 else 120]
    best = (None, '', 0.0, None)
    for name, s in seqs.items():
        if not s: continue
        aln = pairwise2.align.globalxs(variable, s, -2, -1, one_alignment_only=True)
        if not aln: continue
        t, q, score, st, en = aln[0]
        matches = sum(1 for a, b in zip(t, q) if a == b and a != '-')
        ident = matches / max(1, sum(1 for a in t if a != '-'))
        if ident > best[2]:
            best = (name, s, ident, aln[0])
    return best

for label, seqs in [('greedy', greedy_seqs), ('dbg', dbg_seqs)]:
    name, seq, ident, aln = best_pick(seqs)
    print(f'\n=== {label.upper()} best scaffold for {SAMPLE_NAME}: {name}')
    print(f'    identity to {SAMPLE_NAME} variable domain: {ident:.1%}\n')
    if aln:
        print(format_alignment(*aln, full_sequences=False)[:2200])

## 11. (Optional stretch) Close the loop with ESM-2

From Module 1: take the best scaffold and the true reference, embed both with ESM-2, compare. If the assembly is mostly right outside CDR3, the embeddings should be very close.

Skip this cell if you're short on time.

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModel

    MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).eval()

    @torch.no_grad()
    def embed(seq):
        x = tokenizer(seq, return_tensors='pt')
        h = model(**x).last_hidden_state[0, 1:-1].mean(0).numpy()
        return h / ___    # TODO_8: to make a dot product equal cosine similarity, normalise the vector by its L2 norm. Use `np.linalg.norm(h)`.

    ref_emb = embed(NB_REF)
    best_name, best_seq, _, _ = best_pick(dbg_seqs)
    if best_seq:
        scaf_emb = embed(best_seq)
        cos = float(ref_emb @ scaf_emb)
        print(f'Cosine similarity of best DBG scaffold to {SAMPLE_NAME} reference: {cos:.4f}')
        print('(1.0 = identical, 0.0 = orthogonal; expect 0.95+ on a successful assembly)')
    else:
        print('No DBG scaffold to compare.')
except Exception as e:
    print('Skipping ESM closure step:', e)

## 12. Discussion prompts

1. **Which assembly mode worked better on nb6 — greedy or DBG?** Why might that be? (Hint: k-mer size, error tolerance, peptide redundancy.)
2. **Where did the assembly fail?** Look at the coverage track. Is CDR3 the worst region? Why is it so hard *both* for MS de novo *and* for MSA-based methods (Module 1 slide 10)?
3. **What would you change about the protease panel?** Look back at the `hit_rate` per protease in section 3. Could you have spent your beam time better?
4. **The confidence cutoff** (`--conf 0.9`). What happens if you set it to 0.7 or 0.99? Run a quick sweep at home.
5. **No reference?** In a real nanobody discovery campaign you don't *have* the reference. How would you judge assembly quality without one? (Hint: read agreement between greedy and DBG, embedding plausibility from Module 1, …)
6. **Did the pre-assembly checks (Section 4) predict where the assembly would succeed?** Compare the dark regions in the pre-assembly coverage to the dark regions in the post-assembly coverage. Did InstaNexus close any gaps that the exact-match preview missed?

Bring one observation to the closing discussion.

## End of Exercise 2 — end of the 4-hour module

You have:
- Run a **foundation model** (ESM-2) on protein sequences and used the embeddings for visualisation, classification, zero-shot variant scoring, and a regression task (Exercise 1).
- **Inspected real InstaNovo predictions** on a nanobody MS dataset — log-prob distribution, per-residue confidence on a 0-1 scale, peptide length by protease, inter-protease Jaccard overlap, and a quick pre-assembly coverage check.
- Run **InstaNexus** in both greedy and De Bruijn graph modes on a nanobody (nb6), and pulled the consensus scaffolds out of the actual output layout.
- Compared the assembled scaffolds against the **known reference** and visualised coverage with CDR1/2/3 highlighted.
- **Closed the loop** by re-embedding the best scaffold with ESM-2 and measuring cosine similarity to the reference.

Same paradigm — foundation model pretraining → embeddings or scoring → downstream task — across three different data modalities (protein sequences, mass spectra, and now your own assembly output). That is the foundation-model paradigm in biology.